# Боевой прогон RL-модели

Этот ноутбук запускает поэтапное обучение (`1M -> 3M -> 5M`), собирает метрики holdout по нескольким сидам и строит графики обучения.

Что делает:
- запускает `scripts/train_staged.py`;
- оценивает каждый этап через `scripts/evaluate.py` с `--seeds`;
- собирает статистику из JSON-артефактов;
- строит графики `win_rate / mean_reward / mean_steps / survival_rate`.

Перед запуском проверьте, что зависимости установлены и доступны скрипты в `scripts/`.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = Path("d:/RL DD")

RUNS = ROOT / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"PYTHON={sys.executable}")


def run(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(ROOT))

In [ ]:
# Конфиг боевого прогона
MILESTONES = [1_000_000, 3_000_000, 5_000_000]
SEEDS = [7, 17, 27]
EPISODES_PER_SEED = 120

# Для быстрого sanity-check можно временно заменить на [50_000, 100_000]
# и EPISODES_PER_SEED = 20

In [ ]:
# 1) Поэтапное обучение (1M -> 3M -> 5M)
milestones_arg = ",".join(str(x) for x in MILESTONES)

run([
    sys.executable,
    str(ROOT / "scripts" / "train_staged.py"),
    "--milestones", milestones_arg,
    "--seed", str(SEEDS[0]),
    "--n-envs", "8",
    "--device", "cuda",
    "--learning-rate", "2.5e-4",
    "--lr-end-ratio", "0.08",
    "--ent-coef", "0.01",
    "--net-arch", "384,384",
    "--n-steps", "1024",
    "--batch-size", "256",
    "--n-epochs", "4",
    "--gamma", "0.995",
    "--max-episode-steps", "120",
    "--checkpoint-global-freq", "50000",
    "--eval-global-freq", "25000",
    "--out-prefix", "runs/dd2_ppo_stage",
])

In [ ]:
# 2) Holdout-оценка каждого этапа по нескольким сидам
seeds_arg = ",".join(str(s) for s in SEEDS)
eval_jsons: list[Path] = []

for steps in MILESTONES:
    model_path = ROOT / f"runs/dd2_ppo_stage_{steps}.zip"
    out_json = ROOT / f"runs/eval_dd2_ppo_stage_{steps}.json"
    eval_jsons.append(out_json)
    run([
        sys.executable,
        str(ROOT / "scripts" / "evaluate.py"),
        "--model", str(model_path),
        "--episodes", str(EPISODES_PER_SEED),
        "--seeds", seeds_arg,
        "--out-json", str(out_json),
    ])

print("Saved:")
for p in eval_jsons:
    print(" -", p)

In [ ]:
# 3) Сбор статистики в таблицы
stage_rows = []
encounter_rows = []

for steps in MILESTONES:
    p = ROOT / f"runs/eval_dd2_ppo_stage_{steps}.json"
    payload = json.loads(p.read_text(encoding="utf-8"))
    summary = payload["summary"]
    stage_rows.append({
        "stage_steps": steps,
        "win_rate": summary["win_rate"],
        "mean_reward": summary["mean_reward"],
        "mean_steps": summary["mean_steps"],
        "survival_rate": summary["survival_rate"],
        "seeds": ",".join(str(x) for x in payload.get("seeds", [])),
        "episodes_per_seed": payload.get("episodes_per_seed", None),
    })
    for enc_id, row in payload.get("encounters", {}).items():
        encounter_rows.append({
            "stage_steps": steps,
            "encounter": enc_id,
            "win_rate": row["win_rate"],
            "mean_reward": row["mean_reward"],
            "mean_steps": row["mean_steps"],
            "survival_rate": row["survival_rate"],
        })

df_stage = pd.DataFrame(stage_rows).sort_values("stage_steps").reset_index(drop=True)
df_enc = pd.DataFrame(encounter_rows).sort_values(["encounter", "stage_steps"]).reset_index(drop=True)

df_stage

In [ ]:
# 4) Графики общей динамики по этапам
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df_stage["stage_steps"], df_stage["win_rate"], marker="o")
axes[0, 0].set_title("Win rate")
axes[0, 0].set_xlabel("Timesteps")
axes[0, 0].set_ylabel("Rate")
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(df_stage["stage_steps"], df_stage["mean_reward"], marker="o")
axes[0, 1].set_title("Mean reward")
axes[0, 1].set_xlabel("Timesteps")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(df_stage["stage_steps"], df_stage["mean_steps"], marker="o")
axes[1, 0].set_title("Mean episode steps")
axes[1, 0].set_xlabel("Timesteps")
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(df_stage["stage_steps"], df_stage["survival_rate"], marker="o")
axes[1, 1].set_title("Survival rate")
axes[1, 1].set_xlabel("Timesteps")
axes[1, 1].set_ylabel("Rate")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 5) Графики по каждому holdout-encounter
for metric in ["win_rate", "mean_reward", "mean_steps", "survival_rate"]:
    pivot = df_enc.pivot(index="stage_steps", columns="encounter", values=metric)
    ax = pivot.plot(marker="o", figsize=(12, 5), title=f"{metric} by encounter")
    ax.set_xlabel("Timesteps")
    ax.grid(True, alpha=0.3)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6) Сохранение сводных CSV для дальнейшего анализа
stage_csv = ROOT / "runs" / "stage_summary.csv"
enc_csv = ROOT / "runs" / "stage_encounter_summary.csv"

df_stage.to_csv(stage_csv, index=False, encoding="utf-8")
df_enc.to_csv(enc_csv, index=False, encoding="utf-8")

print("Saved:")
print(" -", stage_csv)
print(" -", enc_csv)